# Этап 9: финальные материалы для отчета

Ноутбук собирает итоговые таблицы сравнения, рисунки и Markdown для отчета. Новые эксперименты здесь не запускаются.


In [1]:
from __future__ import annotations

import os
import subprocess
import sys
from pathlib import Path
from urllib.parse import quote, urlsplit, urlunsplit

from google.colab import drive

try:
    from google.colab import userdata
except Exception:
    userdata = None

DRIVE_ROOT = Path('/content/drive/MyDrive/diploma/petr_text_to_visualization_part')
REPO_ROOT = Path('/content/petukhov_t2v_repo')
REPO_URL = 'https://github.com/petrussia/NL2BI-AI-assistant.git'
BRANCH = 'experiments/peter'
TOKEN_FILE = DRIVE_ROOT / 'secrets' / 'github_token.txt'

def read_token() -> str | None:
    if userdata is not None:
        try:
            token = userdata.get('GITHUB_TOKEN')
            if token:
                return token.strip()
        except Exception:
            pass
    if TOKEN_FILE.exists():
        return TOKEN_FILE.read_text(encoding='utf-8').strip()
    return None

def auth_url(token: str | None) -> str:
    if not token:
        return REPO_URL
    parts = urlsplit(REPO_URL)
    return urlunsplit((parts.scheme, f'x-access-token:{quote(token, safe="")}@{parts.netloc}', parts.path, parts.query, parts.fragment))

def run(cmd: list[str], cwd: Path | None = None) -> None:
    shown = ['***' if 'x-access-token:' in part else part for part in cmd]
    print('RUN', ' '.join(shown))
    subprocess.check_call(cmd, cwd=str(cwd) if cwd else None)

drive.mount('/content/drive', force_remount=False)
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
os.environ['T2V_DRIVE_ROOT'] = str(DRIVE_ROOT)
os.environ['T2V_REPO_ROOT'] = str(REPO_ROOT)

repo_url = auth_url(read_token())
if not REPO_ROOT.exists():
    run(['git', 'clone', '--branch', BRANCH, '--single-branch', repo_url, str(REPO_ROOT)])
else:
    run(['git', 'remote', 'set-url', 'origin', repo_url], cwd=REPO_ROOT)
    run(['git', 'fetch', 'origin', BRANCH], cwd=REPO_ROOT)
    run(['git', 'checkout', BRANCH], cwd=REPO_ROOT)
    try:
        run(['git', 'pull', '--ff-only', 'origin', BRANCH], cwd=REPO_ROOT)
    except subprocess.CalledProcessError:
        print('git pull --ff-only failed; resetting Colab checkout to origin branch')
        run(['git', 'reset', '--hard', f'origin/{BRANCH}'], cwd=REPO_ROOT)
run(['git', 'remote', 'set-url', 'origin', REPO_URL], cwd=REPO_ROOT)
run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO_ROOT / 'requirements.txt')])
run([sys.executable, '-m', 'pip', 'install', '-q', 'pillow', 'openpyxl', 'matplotlib'])
run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO_ROOT)])
run([sys.executable, '-c', 'import t2v_eval; print(t2v_eval.__version__)'])
print('STAGE9_SETUP_OK')


Mounted at /content/drive
RUN git clone --branch experiments/peter --single-branch *** /content/petukhov_t2v_repo
RUN git remote set-url origin https://github.com/petrussia/NL2BI-AI-assistant.git
RUN /usr/bin/python3 -m pip install -q -r /content/petukhov_t2v_repo/requirements.txt
RUN /usr/bin/python3 -m pip install -q pillow openpyxl matplotlib
RUN /usr/bin/python3 -m pip install -q -e /content/petukhov_t2v_repo
RUN /usr/bin/python3 -c import t2v_eval; print(t2v_eval.__version__)
STAGE9_SETUP_OK


In [6]:
from __future__ import annotations

import subprocess
import sys
from pathlib import Path

REPO_ROOT = Path('/content/petukhov_t2v_repo')
script_path = REPO_ROOT / 'scripts' / 'make_stage9_report_materials.py'
result = subprocess.run([sys.executable, str(script_path)], cwd=str(REPO_ROOT), text=True, capture_output=True)
print(result.stdout)
if result.stderr:
    print(result.stderr)
if result.returncode != 0:
    raise RuntimeError(f'Сборка материалов этапа 9 завершилась ошибкой {result.returncode}')
print('STAGE9_BUILD_OK')


{
  "output_dir": "/content/petukhov_t2v_repo/reports/stage9_report_materials",
  "drive_root": "/content/drive/MyDrive/diploma/petr_text_to_visualization_part"
}

STAGE9_BUILD_OK


In [7]:
from __future__ import annotations

from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/diploma/petr_text_to_visualization_part')
target = DRIVE_ROOT / 'reports' / 'stage9_report_materials'
required = [
    target / 'stage9_tables.xlsx',
    target / 'tables' / 'comparison_solutions.csv',
    target / 'tables' / 'quality_metrics.csv',
    target / 'tables' / 'latency_memory_failure.csv',
    target / 'tables' / 'applicability.csv',
    target / 'tables' / 'risks_limitations.csv',
    target / 'figures' / 'pipeline_architecture.png',
    target / 'figures' / 'nvbench_postquery_flow.png',
    target / 'figures' / 'key_metrics_comparison.png',
    target / 'figures' / 'system_metrics_comparison.png',
    target / 'figures' / 'examples_grid_gold_vs_predicted.png',
    target / 'latex' / 'quality_metrics_table.tex',
    target / 'practice_report_materials.md',
    target / 'run_inventory.json',
]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError('\n'.join(missing))
print('Папка Drive:', target)
print('Файлы:')
for path in sorted(target.rglob('*')):
    if path.is_file():
        rel = path.relative_to(target)
        print(f'  {rel} ({path.stat().st_size} bytes)')
print('STAGE9_VERIFY_OK')


Папка Drive: /content/drive/MyDrive/diploma/petr_text_to_visualization_part/reports/stage9_report_materials
Файлы:
  comparison_table.md (891 bytes)
  figures/examples_grid_gold_vs_predicted.png (116840 bytes)
  figures/examples_grid_source.txt (100 bytes)
  figures/figure_manifest.json (865 bytes)
  figures/key_metrics_comparison.png (250841 bytes)
  figures/nvbench_postquery_flow.png (56740 bytes)
  figures/pipeline_architecture.png (57050 bytes)
  figures/system_metrics_comparison.png (242999 bytes)
  final_runs.json (1643 bytes)
  latex/quality_metrics_table.pdf (14969 bytes)
  latex/quality_metrics_table.tex (679 bytes)
  practice_report_materials.md (9737 bytes)
  quality_metrics_table.md (491 bytes)
  run_inventory.json (5427 bytes)
  stage9_tables.xlsx (435855 bytes)
  tables/applicability.csv (1485 bytes)
  tables/comparison_solutions.csv (776 bytes)
  tables/latency_memory_failure.csv (217 bytes)
  tables/quality_metrics.csv (340 bytes)
  tables/risks_limitations.csv (1260 by